# LLM from Scratch

Building **nanoGPT** inspired by Andrej Kathaparthy's approach

1. It is initially trained on **Tiny Shakespeare** dataset.

In [41]:
#importing libraries
import torch

### Downloading the data

- Using `!wget` which downloads the files at a given url.
- URL in this case is https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt.
- The file we download is `input.txt`.

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2025-11-13 01:48:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.007s  

2025-11-13 01:48:01 (144 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
#reading the file
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [11]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [10]:
print(len(text))

1115394


### Encoder Decoder

- Encoder is a mapping from character to number.
- Decoder gives the character back based on the number.

This is a simple character level tokenizer.

In [15]:
#creating a sorted list of unique characters that appear and count the total for a vocab size
set_of_text = set(text)
print(set_of_text)
unique_list = list(set_of_text)
print(unique_list)
chars = sorted(unique_list)
print(chars)
vocab_size = len(chars)
print(vocab_size)

{'3', 'h', 'v', 'U', 'm', 'W', 'P', ' ', 'I', 'j', 'V', 'n', '&', 'C', 'A', 'l', 'T', 'H', 'r', 'Y', 'L', 'a', 'b', 'g', 'z', 'u', 'i', 'E', 'X', '.', 'M', 'o', "'", '!', 'F', 'f', 'K', 'y', 'S', 'w', 'R', 'd', 'Z', 'p', '$', 'e', 'B', 'D', 'x', 'O', '\n', 'q', 'J', 'N', 'c', 'G', '?', ';', 'Q', 't', '-', 'k', 's', ':', ','}
['3', 'h', 'v', 'U', 'm', 'W', 'P', ' ', 'I', 'j', 'V', 'n', '&', 'C', 'A', 'l', 'T', 'H', 'r', 'Y', 'L', 'a', 'b', 'g', 'z', 'u', 'i', 'E', 'X', '.', 'M', 'o', "'", '!', 'F', 'f', 'K', 'y', 'S', 'w', 'R', 'd', 'Z', 'p', '$', 'e', 'B', 'D', 'x', 'O', '\n', 'q', 'J', 'N', 'c', 'G', '?', ';', 'Q', 't', '-', 'k', 's', ':', ',']
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [16]:
#or basically
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


### Note

Vocab contains both uppercase and lowercase letters with a few special characters in the beginning.

In [17]:
#simple encoder/decoder
#make a dictionary that has character as key and its place as value
#index for the char is found due to enumerate
stoi = {ch:i for i,ch in enumerate(chars)}
#same but other way round
itos = {i:ch for i,ch in enumerate(chars)}
#encoder - for every character in a given string, return a stoi value or index
encode = lambda s:[stoi[c] for c in s]
#decoder
decode = lambda l:''.join([itos[i] for i in l])

In [28]:
def encoder(str):
  arr = []
  for c in str:
    arr.append(stoi[c])
  return arr

In [39]:
def decoder(arr):
  chars = []
  for num in arr:
    chars.append(itos[num])
  return "".join(chars)

In [29]:
print(encode('is this vocab reAdy?'))

[47, 57, 1, 58, 46, 47, 57, 1, 60, 53, 41, 39, 40, 1, 56, 43, 13, 42, 63, 12]


In [30]:
print(decode([2, 4, 5]))

!&'


In [32]:
print(encoder('is this vocab reAdy?'))

[47, 57, 1, 58, 46, 47, 57, 1, 60, 53, 41, 39, 40, 1, 56, 43, 13, 42, 63, 12]


In [40]:
print(decoder([2, 4, 5]))

!&'


### Creating a tensor for encoded text

A pytorch tensor is a multidimensional array like a numpy array with additional powers for autograd or deep learning.

In [42]:
encoded_text = encode(text)

In [50]:
print(text[:50])

First Citizen:
Before we proceed any further, hear


In [49]:
print(encoded_text[:50])

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56]


### Note

Sanity check: `i` of `First` and `Citizen` are both `47`, so the encoder works.

In [51]:
#making a tensor of the data
data = torch.tensor(encoded_text, dtype=torch.long)

In [52]:
print(data[:50])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


In [55]:
data.shape

torch.Size([1115394])

### Splitting the data

This split is done to keep the test data completely separate.

In [56]:
#training
n = int(0.9*len(data)) #90%
train_data = data[:n]
val_data = data[n:]